In [ ]:
!pip install -U transformers accelerate datasets evaluate pandas openpyxl
!pip install -U bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 145.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 148.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 52.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
   

In [ ]:
import platform, shutil, torch

print("OS:", platform.platform())
print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

if shutil.which("git-lfs") is None:
    print("⚠ Git-LFS is NOT installed or not on PATH.")
    print("  On Windows, open PowerShell as admin and run:")
    print('    winget install Git.Git')
    print('    git lfs install')
else:
    print("✅ Git-LFS is installed.")

OS: Linux-6.6.105+-x86_64-with-glibc2.35
Python: 3.12.12
Torch: 2.9.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Capability: (8, 0)
✅ Git-LFS is installed.


In [ ]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")


import torch
torch.backends.cuda.matmul.allow_tf32 = True
try:
    import torch._dynamo
    torch._dynamo.config.cache_size_limit = 512
except Exception:
    pass

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 2))


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Total VRAM (GB): 85.17


In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional
import math, random, re

import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
)

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
OUTPUT_DIR = "/workspace/modell/data/llama3_8b_radiology_cls"
RANDOM_SEED = 42

MAX_LENGTH = 1024

NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 16
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.05
LOGGING_STEPS = 10

MIN_SAMPLES_PER_CLASS = 50
TEST_SIZE = 0.1
VAL_SIZE = 0.1
EXCEL_PATHS = [
    "/workspace/modell/Data/Traindata.xlsx",
    "radiologue_dataset.xlsx"
]


TEXT_CANDIDATES = [
    "text", "Text", "input", "Input", "prompt", "Prompt",
    "Anamnes", "Anamnes:", "Anamnesis",
    "Protocol, Fragestallning", "Frågeställning", "Frageställning", "Fragesställning",
    "Summa", "Résumé", "Resume", "Summary", "Beskrivning"
]
LABEL_CANDIDATES = [
    "Protocol", "label", "Label", "target", "Target", "class", "Class", "End Result", "EndResult", "end_result", "Frågeställning"
]

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


In [ ]:
def load_excel_first(path_list):
    for p in path_list:
        if os.path.exists(p):
            return pd.read_excel(p)
    raise FileNotFoundError("Excel dataset not found in expected locations. Set EXCEL_PATHS accordingly.")

df_raw = load_excel_first(EXCEL_PATHS)
print("Columns:", list(df_raw.columns))
print("Shape:", df_raw.shape)

def pick_column(cands, df):
    for c in cands:
        if c in df.columns:
            return c
    return None

label_col = pick_column(LABEL_CANDIDATES, df_raw)
anam_col = pick_column(["Anamnes", "Anamnes:", "Anamnesis"], df_raw)
frage_col = pick_column(["Fragestallning", "Frågeställning", "Frageställning", "Fragesställning"], df_raw)
text_col = pick_column(TEXT_CANDIDATES, df_raw)

if anam_col or frage_col:
    df_raw["__text__"] = (
        (df_raw[anam_col].astype(str) if anam_col else "") + "\n" +
        (df_raw[frage_col].astype(str) if frage_col else "")
    ).str.strip()
    TEXT_COLUMN = "__text__"
elif text_col:
    TEXT_COLUMN = text_col
else:

    obj_cols = [c for c in df_raw.columns if df_raw[c].dtype == "object" and c != label_col]
    if not obj_cols:
        raise KeyError("Could not find a suitable text column. Please set TEXT_COLUMN manually.")
    TEXT_COLUMN = obj_cols[0]

if label_col is None:

    candidates = [c for c in df_raw.columns if c != TEXT_COLUMN and df_raw[c].dtype == "object"]
    label_col = None
    for c in candidates:
        nunique = df_raw[c].nunique(dropna=True)
        if 2 <= nunique <= 200:
            label_col = c
            break
    if label_col is None:
        raise KeyError("Could not infer label column. Please rename your label column to one of LABEL_CANDIDATES.")

print("Detected TEXT_COLUMN:", TEXT_COLUMN)
print("Detected LABEL_COLUMN:", label_col)

# Clean rows
df = df_raw[[TEXT_COLUMN, label_col]].dropna()
df = df[(df[TEXT_COLUMN].astype(str).str.len() > 1) & (df[label_col].astype(str).str.len() > 0)]

vc = df[label_col].value_counts()
keep = set(vc[vc >= MIN_SAMPLES_PER_CLASS].index.tolist())
df = df[df[label_col].isin(keep)].copy()
print("Kept classes:", sorted(keep))
print("New shape after filtering:", df.shape)


Columns: ['Anamnes', 'Protocol']
Shape: (1353, 2)
Detected TEXT_COLUMN: __text__
Detected LABEL_COLUMN: Protocol
Kept classes: ['A1', 'A3', 'A4', 'A4 med sen fas', 'B10', 'B2', 'B3', 'BBA1', 'L5', 'L7', 'N1', 'N14', 'N4', 'P2']
New shape after filtering: (1127, 2)


In [ ]:
labels = sorted(df[label_col].astype(str).unique().tolist())
label2id = {y:i for i,y in enumerate(labels)}
id2label = {i:y for y,i in label2id.items()}

def make_prompt(text: str) -> str:
    return (
        "Klassificera följande text till exakt en etikett från listan.\n"
        "Svara ENDAST med etiketten, utan extra text.\n\n"
        f"Tillåtna etiketter: {' | '.join(labels)}\n\n"
        f"Text: {text}\n\nSvar:"
    )

df["__prompt__"] = df[TEXT_COLUMN].astype(str).map(make_prompt)
df["__label__"]  = df[label_col].astype(str)
print("Example prompt:\n", df["__prompt__"].iloc[0][:500], "...")
print("Example label:", df["__label__"].iloc[0])


Example prompt:
 Klassificera följande text till exakt en etikett från listan.
Svara ENDAST med etiketten, utan extra text.

Tillåtna etiketter: A1 | A3 | A4 | A4 med sen fas | B10 | B2 | B3 | BBA1 | L5 | L7 | N1 | N14 | N4 | P2

Text: Anamnes: Kvinna som är utredd för lungföränding via lungmedicin. Man har som bifynd funnit ett bukaortaaneurysm på en PET-CT, men bilderna är av dålig kvalité. Tacksam CT-aorta att göras inom 1 månad med anledning om misstänkt större aneurysm.

Frågeställning:

Svar: ...
Example label: A1


In [ ]:
train_df, temp_df = train_test_split(df, test_size=TEST_SIZE+VAL_SIZE, random_state=RANDOM_SEED, stratify=df["__label__"])
rel_val = VAL_SIZE / (TEST_SIZE + VAL_SIZE)
val_df, test_df   = train_test_split(temp_df, test_size=1-rel_val, random_state=RANDOM_SEED, stratify=temp_df["__label__"])

print("Splits:", len(train_df), len(val_df), len(test_df))

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))
ds = DatasetDict({"train": train_ds, "validation": val_ds, "test": test_ds})


Splits: 901 113 113


In [ ]:
from huggingface_hub import login
login("")

In [ ]:
os.environ["HF_TOKEN"] = ""

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
attn_impl = "sdpa"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    padding_side="right",
    truncation_side="right",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

print(
    "pad_token_id:", tokenizer.pad_token_id,
    "| bos_token_id:", tokenizer.bos_token_id,
    "| eos_token_id:", tokenizer.eos_token_id,
)

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    if major >= 8:
        torch_dtype = torch.bfloat16
    else:
        torch_dtype = torch.float16
else:
    torch_dtype = None

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    attn_implementation=attn_impl,
    use_cache=True,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

print("✅ Llama3 model loaded successfully.")


pad_token_id: 128009 | bos_token_id: 128000 | eos_token_id: 128009


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Meta-Llama-3-8B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Llama3 model loaded successfully.


In [ ]:
from transformers import DataCollatorWithPadding

MAX_LEN = MAX_LENGTH

def build_tokenize_fn(max_length:int):
    def _tok(batch):
        text_list = batch[TEXT_COLUMN] if isinstance(batch[TEXT_COLUMN], list) else [batch[TEXT_COLUMN]]
        enc = tokenizer(
            text_list,
            truncation=True,
            max_length=max_length,
            padding=False  # dynamic padding handled by DataCollatorWithPadding
        )

        labels_scalar = []
        raw_labels = batch["__label__"] if isinstance(batch["__label__"], list) else [batch["__label__"]]
        for y in raw_labels:
            labels_scalar.append(int(label2id[str(y)]))
        enc["labels"] = labels_scalar
        return enc
    return _tok

def retokenize_splits(dataset_dict:DatasetDict, max_length:int) -> DatasetDict:
    tok_fn = build_tokenize_fn(max_length)

    keep_cols = [TEXT_COLUMN, "__label__"]
    cols_to_remove = [c for c in dataset_dict["train"].column_names if c not in keep_cols]
    out = DatasetDict()
    for split in dataset_dict.keys():
        ds_ = dataset_dict[split]
        ds_ = ds_.remove_columns(cols_to_remove)
        ds_ = ds_.map(tok_fn, batched=True, desc=f"Tokenizing {split} @ max_length={max_length}")
        out[split] = ds_
    return out

tokenized = retokenize_splits(ds, MAX_LENGTH)
print(tokenized)


Tokenizing train @ max_length=1024:   0%|          | 0/901 [00:00<?, ? examples/s]

Tokenizing validation @ max_length=1024:   0%|          | 0/113 [00:00<?, ? examples/s]

Tokenizing test @ max_length=1024:   0%|          | 0/113 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['__text__', '__label__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 901
    })
    validation: Dataset({
        features: ['__text__', '__label__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 113
    })
    test: Dataset({
        features: ['__text__', '__label__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 113
    })
})


In [ ]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

optim_choice = "adamw_torch_fused"
try:
    import bitsandbytes as bnb  # noqa: F401
    optim_choice = "adamw_bnb_8bit"
except Exception:
    pass

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True if torch.cuda.is_available() else False,
    optim=optim_choice,
    lr_scheduler_type="cosine",
    report_to="none",
    dataloader_pin_memory=True,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)


/tmp/ipython-input-3291267980.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
import gc, traceback, torch

def try_train():
    global trainer, tokenized, MAX_LENGTH
    attempts = [
        {"max_len": MAX_LENGTH, "bs": args.per_device_train_batch_size, "accum": args.gradient_accumulation_steps},
        {"max_len": 768, "bs": 1, "accum": max(24, args.gradient_accumulation_steps)},
        {"max_len": 512, "bs": 1, "accum": max(32, args.gradient_accumulation_steps)},
        {"max_len": 384, "bs": 1, "accum": max(40, args.gradient_accumulation_steps)},
    ]

    for i, cfg in enumerate(attempts, 1):
        print(f"\n=== Attempt {i}/{len(attempts)}: max_len={cfg['max_len']}, bs={cfg['bs']}, accum={cfg['accum']} ===")
        try:
            if cfg["max_len"] != MAX_LENGTH:
                MAX_LENGTH = cfg["max_len"]
                tokenized = retokenize_splits(ds, MAX_LENGTH)
                trainer.train_dataset = tokenized["train"]
                trainer.eval_dataset = tokenized["validation"]

            trainer.args.per_device_train_batch_size = cfg["bs"]
            trainer.args.gradient_accumulation_steps = cfg["accum"]
            trainer.args.num_train_epochs = NUM_EPOCHS
            trainer.args.learning_rate = LEARNING_RATE

            trainer.args.evaluation_strategy = "epoch"
            trainer.args.save_strategy = "epoch"
            trainer.args.eval_delay = 0
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

            print("Starting training…")
            result = trainer.train()
            print("Best checkpoint:", trainer.state.best_model_checkpoint)
            return result
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("⚠️ OOM encountered. Retrying with tighter settings…")
                traceback.print_exc()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()
                continue
            else:
                raise
    raise RuntimeError("All OOM backoff attempts failed. Lower MAX_LENGTH further or enable ZeRO‑2/3.")

train_result = try_train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)



=== Attempt 1/4: max_len=1024, bs=2, accum=16 ===
Starting training…


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,38.022400,0.913770
2,6.366600,0.403816
3,2.060300,0.402772


Best checkpoint: None


('/workspace/modell/data/llama3_8b_radiology_cls/tokenizer_config.json',
 '/workspace/modell/data/llama3_8b_radiology_cls/special_tokens_map.json',
 '/workspace/modell/data/llama3_8b_radiology_cls/chat_template.jinja',
 '/workspace/modell/data/llama3_8b_radiology_cls/tokenizer.json')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

eval_split = "test" if "test" in tokenized else "validation"
eval_ds = tokenized[eval_split]
pred_out = trainer.predict(eval_ds)
logits = pred_out.predictions
y_pred = np.argmax(logits, axis=-1)
y_true = np.array(eval_ds["labels"])

# Basic metrics
acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average="macro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")

print(f"Eval split: {eval_split}")
print(f"Accuracy: {acc:.4f}")
print(f"F1 (macro): {f1_macro:.4f}")
print(f"F1 (weighted): {f1_weighted:.4f}")

# Detailed per-class report
target_names = [id2label[i] for i in range(len(id2label))]
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(id2label))))
cm_df = pd.DataFrame(cm, index=[f"true_{t}" for t in target_names],
                        columns=[f"pred_{t}" for t in target_names])
print("\nConfusion matrix:")
display(cm_df)

texts = eval_ds[TEXT_COLUMN]
true_lbls = [id2label[int(i)] for i in y_true]
pred_lbls = [id2label[int(i)] for i in y_pred]

pred_df = pd.DataFrame({
    "text": texts,
    "true_label": true_lbls,
    "pred_label": pred_lbls
})

out_csv = f"{OUTPUT_DIR}/predictions_{eval_split}.csv"
pred_df.to_csv(out_csv, index=False)
print(f"\nSaved predictions to: {out_csv}")


Eval split: test
Accuracy: 0.9204
F1 (macro): 0.9227
F1 (weighted): 0.9192

Classification report:
                precision    recall  f1-score   support

            A1     0.8750    1.0000    0.9333         7
            A3     0.8000    0.6667    0.7273         6
            A4     0.7143    0.7143    0.7143         7
A4 med sen fas     1.0000    1.0000    1.0000         5
           B10     1.0000    1.0000    1.0000         6
            B2     1.0000    1.0000    1.0000         7
            B3     0.9600    0.8571    0.9057        28
          BBA1     1.0000    1.0000    1.0000         6
            L5     0.9000    1.0000    0.9474         9
            L7     1.0000    1.0000    1.0000         7
            N1     1.0000    1.0000    1.0000         7
           N14     0.8571    1.0000    0.9231         6
            N4     0.8571    0.8571    0.8571         7
            P2     0.8333    1.0000    0.9091         5

      accuracy                         0.9204       113
   

,pred_A1,pred_A3,pred_A4,pred_A4 med sen fas,pred_B10,pred_B2,pred_B3,pred_BBA1,pred_L5,pred_L7,pred_N1,pred_N14,pred_N4,pred_P2
true_A1,7,0,0,0,0,0,0,0,0,0,0,0,0,0
true_A3,0,4,2,0,0,0,0,0,0,0,0,0,0,0
true_A4,1,1,5,0,0,0,0,0,0,0,0,0,0,0
true_A4 med sen fas,0,0,0,5,0,0,0,0,0,0,0,0,0,0
true_B10,0,0,0,0,6,0,0,0,0,0,0,0,0,0
true_B2,0,0,0,0,0,7,0,0,0,0,0,0,0,0
true_B3,0,0,0,0,0,0,24,0,1,0,0,1,1,1
true_BBA1,0,0,0,0,0,0,0,6,0,0,0,0,0,0
true_L5,0,0,0,0,0,0,0,0,9,0,0,0,0,0
true_L7,0,0,0,0,0,0,0,0,0,7,0,0,0,0



Saved predictions to: /workspace/modell/data/llama3_8b_radiology_cls/predictions_test.csv


In [ ]:
import os, json, datetime, platform, zipfile
import numpy as np
import pandas as pd

try:
    from sklearn.metrics import (
        accuracy_score, f1_score, precision_score, recall_score,
        classification_report, confusion_matrix
    )
except Exception as e:
    raise RuntimeError("scikit-learn is required. Install with: pip install scikit-learn") from e

try:
    import matplotlib.pyplot as plt
    _HAVE_PLT = True
except Exception:
    _HAVE_PLT = False

OUT_DIR = OUTPUT_DIR
os.makedirs(OUT_DIR, exist_ok=True)

trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

if getattr(model.config, "id2label", None):
    _i2l = model.config.id2label
    id2label = {int(k): str(v) for k, v in ( _i2l.items() if isinstance(_i2l, dict) else enumerate(_i2l) )}
elif "id2label" in globals():
    id2label = {int(k): str(v) for k, v in id2label.items()}
else:
    all_ids = set()
    for s in ["train", "validation", "test"]:
        if s in tokenized:
            all_ids.update(set(tokenized[s]["labels"]))
    id2label = {int(i): f"label_{int(i)}" for i in sorted(all_ids)}

label2id = {lbl: int(i) for i, lbl in id2label.items()}
num_labels = len(id2label)

with open(os.path.join(OUT_DIR, "label_mapping.json"), "w", encoding="utf-8") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, ensure_ascii=False, indent=2)

def split_stats(name, ds_split):
    y = np.array(ds_split["labels"])
    ids, counts = np.unique(y, return_counts=True)
    total = int(counts.sum())
    rows = []
    for cid, cnt in zip(ids, counts):
        rows.append({
            "label_id": int(cid),
            "label": id2label[int(cid)],
            "count": int(cnt),
            "proportion": float(cnt / total) if total else 0.0
        })
    return {"name": name, "n": total, "by_class": rows}

splits_present = [k for k in ["train", "validation", "test"] if k in tokenized]
split_info = [split_stats(k, tokenized[k]) for k in splits_present]

with open(os.path.join(OUT_DIR, "split_stats.json"), "w", encoding="utf-8") as f:
    json.dump(split_info, f, ensure_ascii=False, indent=2)

rows_csv = []
for s in split_info:
    for r in s["by_class"]:
        rows_csv.append({"split": s["name"], **r})
pd.DataFrame(rows_csv).to_csv(os.path.join(OUT_DIR, "split_stats.csv"), index=False)

def topk_accuracy(logits: np.ndarray, y_true: np.ndarray, k: int = 3) -> float:
    if logits.ndim != 2:
        logits = np.asarray(logits)
        logits = logits.reshape((logits.shape[0], -1))
    k = int(min(k, logits.shape[1]))
    topk_idx = np.argpartition(logits, -k, axis=1)[:, -k:]
    correct = 0
    for i, true_id in enumerate(y_true):
        if true_id in topk_idx[i]:
            correct += 1
    return correct / len(y_true) if len(y_true) else 0.0

def eval_and_save(split_name: str):
    pred_out = trainer.predict(tokenized[split_name])
    logits = pred_out.predictions
    y_pred = np.argmax(logits, axis=-1)
    y_true = np.array(tokenized[split_name]["labels"])

    acc = float(accuracy_score(y_true, y_pred))
    f1_macro = float(f1_score(y_true, y_pred, average="macro"))
    f1_weighted = float(f1_score(y_true, y_pred, average="weighted"))
    prec_macro = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    prec_weighted = float(precision_score(y_true, y_pred, average="weighted", zero_division=0))
    rec_macro = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    rec_weighted = float(recall_score(y_true, y_pred, average="weighted", zero_division=0))
    top3 = float(topk_accuracy(logits, y_true, k=3))

    target_names = [id2label[i] for i in range(num_labels)]
    clf_rep_txt = classification_report(y_true, y_pred, target_names=target_names, digits=4, zero_division=0)
    with open(os.path.join(OUT_DIR, f"classification_report_{split_name}.txt"), "w", encoding="utf-8") as f:
        f.write(clf_rep_txt)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_labels)))
    cm_df = pd.DataFrame(cm,
                         index=[f"true_{t}" for t in target_names],
                         columns=[f"pred_{t}" for t in target_names])
    cm_csv_path = os.path.join(OUT_DIR, f"confusion_matrix_{split_name}.csv")
    cm_df.to_csv(cm_csv_path)

    cm_png_path = None
    if _HAVE_PLT:
        plt.figure(figsize=(max(6, num_labels * 0.6), max(5, num_labels * 0.6)))
        plt.imshow(cm, interpolation='nearest')
        plt.title(f'Confusion Matrix — {split_name}')
        plt.xlabel('Predicted label')
        plt.ylabel('True label')
        plt.xticks(ticks=np.arange(num_labels), labels=target_names, rotation=90)
        plt.yticks(ticks=np.arange(num_labels), labels=target_names)
        # annotate counts
        for i in range(num_labels):
            for j in range(num_labels):
                plt.text(j, i, str(cm[i, j]), ha="center", va="center")
        plt.tight_layout()
        cm_png_path = os.path.join(OUT_DIR, f"confusion_matrix_{split_name}.png")
        plt.savefig(cm_png_path, dpi=150)
        plt.close()

    texts = None
    if "TEXT_COLUMN" in globals() and TEXT_COLUMN in tokenized[split_name].column_names:
        texts = tokenized[split_name][TEXT_COLUMN]
    pred_df = pd.DataFrame({
        "text": texts if texts is not None else [None] * len(y_true),
        "true_label": [id2label[int(i)] for i in y_true],
        "pred_label": [id2label[int(i)] for i in y_pred],
        "true_id": [int(i) for i in y_true],
        "pred_id": [int(i) for i in y_pred],
    })
    pred_csv_path = os.path.join(OUT_DIR, f"predictions_{split_name}.csv")
    pred_df.to_csv(pred_csv_path, index=False)

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "precision_macro": prec_macro,
        "precision_weighted": prec_weighted,
        "recall_macro": rec_macro,
        "recall_weighted": rec_weighted,
        "top3_accuracy": top3,
        "classification_report_path": f"classification_report_{split_name}.txt",
        "confusion_matrix_csv": os.path.basename(cm_csv_path),
        "confusion_matrix_png": os.path.basename(cm_png_path) if cm_png_path else None,
        "predictions_csv": os.path.basename(pred_csv_path),
        "n_examples": int(len(y_true)),
    }

metrics = {}
for split_name in ["validation", "test"]:
    if split_name in splits_present:
        metrics[split_name] = eval_and_save(split_name)

import torch, transformers
args_info = {
    "model_name_or_path": getattr(model.config, "_name_or_path", None),
    "num_labels": int(getattr(model.config, "num_labels", num_labels)),
    "max_length": int(MAX_LENGTH) if "MAX_LENGTH" in globals() else None,
    "per_device_train_batch_size": int(trainer.args.per_device_train_batch_size),
    "gradient_accumulation_steps": int(trainer.args.gradient_accumulation_steps),
    "num_train_epochs": float(trainer.args.num_train_epochs),
    "learning_rate": float(trainer.args.learning_rate),
    "attn_implementation": str(getattr(model.config, "attn_implementation", None)),
    "gradient_checkpointing": bool(getattr(model, "is_gradient_checkpointing", False)),
    "use_cache": bool(getattr(model.config, "use_cache", True)),
    "best_checkpoint": trainer.state.best_model_checkpoint,
}

env = {
    "timestamp": datetime.datetime.now().isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

report = {
    "labels": id2label,
    "split_info": split_info,
    "metrics": metrics,
    "args": args_info,
    "env": env,
}
with open(os.path.join(OUT_DIR, "run_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

md = []
md.append("# Run Report")
md.append(f"- **Time:** {env['timestamp']}")
md.append(f"- **Model:** {args_info['model_name_or_path']}")
md.append(f"- **Num labels:** {args_info['num_labels']}")
md.append(f"- **Max length:** {args_info['max_length']}")
md.append(f"- **Epochs:** {args_info['num_train_epochs']}")
md.append(f"- **LR:** {args_info['learning_rate']}")
md.append(f"- **Batch size (per device):** {args_info['per_device_train_batch_size']}")
md.append(f"- **Grad accumulation:** {args_info['gradient_accumulation_steps']}")
md.append(f"- **Attention impl:** {args_info['attn_implementation']}")
md.append(f"- **Gradient checkpointing:** {args_info['gradient_checkpointing']}")
md.append(f"- **use_cache:** {args_info['use_cache']}")
md.append(f"- **Best checkpoint:** {args_info['best_checkpoint']}")

md.append("\n## Labels")
for i in range(num_labels):
    md.append(f"- {i}: {id2label[i]}")

md.append("\n## Split Stats")
for s in split_info:
    md.append(f"\n### {s['name']} (n={s['n']})")
    for r in s["by_class"]:
        md.append(f"- {r['label']} (id={r['label_id']}): {r['count']} ({r['proportion']:.2%})")

for split_name in ["validation", "test"]:
    if split_name in metrics:
        m = metrics[split_name]
        md.append(f"\n## {split_name.capitalize()} Metrics (n={m['n_examples']})")
        md.append(f"- Accuracy: **{m['accuracy']:.4f}**")
        md.append(f"- F1 (macro): **{m['f1_macro']:.4f}**, F1 (weighted): **{m['f1_weighted']:.4f}**")
        md.append(f"- Precision (macro): **{m['precision_macro']:.4f}**, Precision (weighted): **{m['precision_weighted']:.4f}**")
        md.append(f"- Recall (macro): **{m['recall_macro']:.4f}**, Recall (weighted): **{m['recall_weighted']:.4f}**")
        md.append(f"- Top-3 accuracy: **{m['top3_accuracy']:.4f}**")
        md.append(f"- Files: `predictions_{split_name}.csv`, `confusion_matrix_{split_name}.csv`, `classification_report_{split_name}.txt`")
        if m.get("confusion_matrix_png"):
            md.append(f"![Confusion Matrix — {split_name}]({m['confusion_matrix_png']})")

with open(os.path.join(OUT_DIR, "run_report.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(md))
zip_path = OUT_DIR.rstrip("/\\") + ".zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            fp = os.path.join(root, fn)
            zf.write(fp, arcname=os.path.relpath(fp, OUT_DIR))

print(f"✅ Saved model, metrics, and reports in: {OUT_DIR}")
print(f"📦 Zipped artifact: {zip_path}")

In [ ]:
!pip -q install markdown weasyprint

from markdown import markdown
from weasyprint import HTML

md_file = f"{OUTPUT_DIR}/run_report.md"
pdf_file = f"{OUTPUT_DIR}/run_report.pdf"
with open(md_file, "r", encoding="utf-8") as f:
    html_content = markdown(f.read())

HTML(string=html_content).write_pdf(pdf_file)

print("Saved PDF to:", pdf_file)
